# 🛡️ EXODUS FLEET VALIDATOR — Test End-to-End

> Validation complète de la chaîne de production U00 → U06

### Mode d'emploi
1. **Cellule 1** : Monte le Drive et configure le chemin
2. **Cellule 2** : Lance la validation complète (3 couches)
3. **Cellule 3** : (Si erreurs) Tente les réparations automatiques
4. **Cellule 4** : (Si tout passe) Génère le certificat de scellement

In [ ]:
#@title 🔗 CELLULE 1 — Setup
#@markdown Monte le Drive et prépare le validateur.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DRIVE_EXODUS_V2"  #@param {type:"string"}

import sys, os
# Cloner le repo si besoin (pour accéder aux scripts)
REPO_LOCAL = "/content/EXODUS-V2"
if not os.path.exists(REPO_LOCAL):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kioka8877-ux/EXODUS-V2.git", REPO_LOCAL], check=True)

sys.path.insert(0, REPO_LOCAL)
from EXO_FLEET_VALIDATOR import FleetValidator

validator = FleetValidator(DRIVE_ROOT, verbose=True)
print("✅ Validateur prêt")

In [ ]:
#@title 🩺 CELLULE 2 — Validation Complète (3 couches)
#@markdown Lance la validation E2E de toute la flotte.

report = validator.validate_full()

# Afficher résumé
print("\n" + "═" * 50)
print("📊 RÉSUMÉ")
print("═" * 50)
for unit in ["U00", "U01", "U02", "U03", "U04", "U05", "U06"]:
    r = report["units"][unit]
    icon = "✅" if r["status"] == "PASS" else "❌"
    print(f"  {icon} {unit} — {r['name']}")

seal = report["fleet_seal"]["status"]
print(f"\n🛡️ Fleet Seal : {seal}")
if seal == "REJECTED":
    print(f"   Raison : {report['fleet_seal']['reason']}")

# Sauvegarder rapport
import json
report_path = f"{DRIVE_ROOT}/FLEET_VALIDATION_REPORT.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"\n📄 Rapport sauvegardé : {report_path}")

In [ ]:
#@title 🔧 CELLULE 3 — Réparations Automatiques
#@markdown Tente de corriger les problèmes détectés.
#@markdown ⚠️ Ne résout que les problèmes structurels (dossiers, liens).
#@markdown Les frégates en échec doivent être re-exécutées manuellement.

fixes = validator.fix()
for fix_msg in fixes["fixes_applied"]:
    print(f"  🔧 {fix_msg}")

if not fixes["fixes_applied"]:
    print("  ✅ Rien à réparer")
else:
    print(f"\n🔧 {len(fixes['fixes_applied'])} réparation(s) effectuée(s)")
    print("   Relancez la Cellule 2 pour re-valider")

In [ ]:
#@title 🛡️ CELLULE 4 — Générer le Fleet Seal Certificate
#@markdown Génère le certificat UNIQUEMENT si la validation est passée à 100%.

cert_path = f"{DRIVE_ROOT}/FLEET_SEAL_CERTIFICATE.md"
success = validator.generate_seal_certificate(cert_path)

if success:
    print(f"\n🛡️ L'Empire EXODUS est officiellement SCELLÉ")
    print(f"   Certificat : {cert_path}")
else:
    print(f"\n❌ Impossible de sceller — corrigez les erreurs d'abord (Cellule 3)")